In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from numpy.linalg import eig
import pandas as pd

#import matplotlib  
#matplotlib.use('Agg')  # Use a non-GUI backend

In [2]:
#cat='C1'
#path=f'C:/shehani/postdoc_work/ML_Diffusion/cal_md/new_cals/{cat}_rel/'
#path=f'C:/shehani/postdoc_work/ML_Diffusion/cal_md/new_cals/{cat}_rel/'

#df = pd.DataFrame(columns=['nrep', '#hops' 'Dnonhop_mean', 'Dnonhop_se', 'D_hop', 'Dtot'])

In [3]:
def oh_xyz(cat, nrep, nsteps):
    #nrep = number of replicas
    #nsteps = number of steps in fs/MD_freq (ex: for 10 ps simulation with MD_freq=10, nsteps=10,000/10=1000)
    path=f'C:/shehani/postdoc_work/ML_Diffusion/cal_md/new_cals/{cat}_rel/'
    #extracting x,y, z cordinates from oh_id.dat files
    x_oh = np.zeros((nrep,nsteps))
    y_oh = np.zeros((nrep,nsteps))
    z_oh = np.zeros((nrep,nsteps))
    indx_oh= np.zeros((nrep,nsteps))

    for i in range(nrep):
        with open(path+ f'oh_id_{i+1}.dat', 'r') as oh_id:
            xyz= oh_id.readlines()[:nsteps]
            
            for j in range(len(xyz)):   
                indx_oh[i,j]=int(xyz[j].split()[1])
                x_oh[i,j]=float(xyz[j].split()[2])
                y_oh[i,j]=float(xyz[j].split()[3])
                z_oh[i,j]=float(xyz[j].split()[4])
                
    #dtime=  np.arange(0.01, (0.01*ndt+0.01), 0.01)
    return(x_oh, y_oh, z_oh, indx_oh)




In [4]:
nrep=15
cat='c2'
nsteps=20000

x_oh, y_oh, z_oh, indx_oh= oh_xyz(cat, nrep, nsteps)

In [5]:
path=f'C:/shehani/postdoc_work/ML_Diffusion/cal_md/new_cals/{cat}_rel/'
d_rep = pd.DataFrame(columns=['nrep', '#hops', 'D_nonhop (mean)', 'D_nonhop (SE)', 'D_hop', 'D_tot'])
d_rep = d_rep.astype(float)
for kk in range(1,nrep+1):
    dtime = np.zeros(nsteps)
    D1x=np.zeros(kk)
    D1y=np.zeros(kk)
    D1=np.zeros(kk)
    rx=[]
    ry=[]
    rz=[]
    t=[]
    ax=[]
    ay=[]
    t_hop=[]

    nhop=0
    for ll in range(kk):
        int_indx= indx_oh[ll,0]
        int_x= x_oh[ll,0]
        int_y= y_oh[ll,0]
        int_z= z_oh[ll,0]
        int_t=0
        hop=0
        for jj in range(nsteps):
            dtime[jj]= np.round(jj*0.01, 2)
            
            if indx_oh[ll,jj]==int_indx:
                pass
                #print('nonhop',ll, jj, indx_oh[ll,jj],dtime[jj], x_oh[ll,jj])
            else:
                hop=hop+1
                #t_hop.append(dtime[jj])
                rx.append((x_oh[ll,jj-1]-int_x)**2)
                ry.append((y_oh[ll,jj-1]-int_y)**2)
                t.append(dtime[jj-1]-int_t)
                #print('hop',ll, jj, indx_oh[ll,jj])
                #print(int_t, int_indx, int_x, int_y)
                #print(dtime[jj-1], indx_oh[ll,jj-1], x_oh[ll,jj-1], y_oh[ll,jj-1])     
                #print(t[-1], rx[-1], ry[-1])   
                ax.append((x_oh[ll,jj]-x_oh[ll,jj-1])**2)
                ay.append((y_oh[ll,jj]-y_oh[ll,jj-1])**2)
                #print(jj)
    
                if hop>1:
                    #ax.append((x_oh[ll,jj]-x_oh[ll,jj-1])**2)
                    #ay.append((y_oh[ll,jj]-y_oh[ll,jj-1])**2)
                    #print(ll,jj,(x_oh[ll,jj]-x_oh[ll,jj-1])**2 )
                    #print(x_oh[ll,jj],x_oh[ll,jj-1])
                    #print(dtime[jj-1],int_t)
                    t_hop.append(dtime[jj-1]-int_t)
    
                
                int_indx= indx_oh[ll,jj]
                int_x= x_oh[ll,jj]
                int_y= y_oh[ll,jj]
                int_z= z_oh[ll,jj]
                int_t=dtime[jj]
                #print(hop,int_t)
    
                #print(indx_oh[ll,jj], x_oh[ll,jj], x_oh[ll,jj-1], dtime[jj],int_t)
        nhop= nhop+hop      
        #print(ll,hop)
        rx.append((x_oh[ll,-1]-int_x)**2)
        ry.append((y_oh[ll,-1]-int_y)**2)
        rz.append((z_oh[ll,-1]-int_z)**2)
        t.append(dtime[-1]-int_t)
        
        D1x[ll]=(np.mean(rx))/(2*np.mean(t))
        D1y[ll]=(np.mean(ry))/(2*np.mean(t))
        D1[ll]=(np.mean(rx)+np.mean(ry))/(4*np.mean(t))
    
    dmean= np.mean(D1)
    dstd= np.std(D1)
    std_err= dstd/np.sqrt(kk)
    
    D2x=(np.mean(ax))/(2*np.mean(t_hop))
    D2y=(np.mean(ay))/(2*np.mean(t_hop))
    D2=(np.mean(ax)+np.mean(ay))/(4*np.mean(t_hop))
    D=dmean+D2

    d_rep.loc[kk,'nrep']=kk
    d_rep.loc[kk,'#hops']= nhop
    d_rep.loc[kk,'D_nonhop (mean)']=dmean
    d_rep.loc[kk,'D_nonhop (SE)']=std_err
    d_rep.loc[kk,'D_hop']=D2
    d_rep.loc[kk,'D_tot']=D 
    print(kk, D1)
    print('nohops:',dmean, std_err)
    print('hops:',D2)
    print('Total:',D)
    print(nhop)
    #print(rx)
    #print(t)
d_rep.to_csv(f'{path}newD_{cat}.csv', index=False)

1 [0.6753325]
nohops: 0.6753324958592607 0.0
hops: 0.16482314113900765
Total: 0.8401556369982683
26
2 [0.6753325  0.96906432]
nohops: 0.8221984071728365 0.10384988181497161
hops: 0.14171416166121117
Total: 0.9639125688340477
48
3 [0.6753325  0.96906432 0.88029664]
nohops: 0.841564484248906 0.07101600872716568
hops: 0.13148543558823128
Total: 0.9730499198371372
70
4 [0.6753325  0.96906432 0.88029664 0.94842682]
nohops: 0.8682800686425465 0.058070071279100265
hops: 0.15346913493373238
Total: 1.0217492035762787
104
5 [0.6753325  0.96906432 0.88029664 0.94842682 1.00241014]
nohops: 0.8951060823893602 0.05228645374268229
hops: 0.15459431487914452
Total: 1.0497003972685046
131
6 [0.6753325  0.96906432 0.88029664 0.94842682 1.00241014 0.99084753]
nohops: 0.9110629905289844 0.04594245151573796
hops: 0.1465256613898001
Total: 1.0575886519187845
150
7 [0.6753325  0.96906432 0.88029664 0.94842682 1.00241014 0.99084753
 0.93766031]
nohops: 0.9148626079217702 0.0395360534309519
hops: 0.138315741185

In [6]:

# dtime = np.zeros(nsteps)
# D1x=np.zeros(nrep)
# D1y=np.zeros(nrep)
# D1=np.zeros(nrep)
# rx=[]
# ry=[]
# rz=[]
# t=[]
# ax=[]
# ay=[]
# t_hop=[]

# nhop=0
# for ll in range(nrep):
#     int_indx= indx_oh[ll,0]
#     int_x= x_oh[ll,0]
#     int_y= y_oh[ll,0]
#     int_z= z_oh[ll,0]
#     int_t=0
#     hop=0
#     for jj in range(nsteps):
#         dtime[jj]= np.round(jj*0.01, 2)
        
#         if indx_oh[ll,jj]==int_indx:
#             pass
#             #print('nonhop',ll, jj, indx_oh[ll,jj],dtime[jj], x_oh[ll,jj])
#         else:
#             hop=hop+1
#             #t_hop.append(dtime[jj])
#             rx.append((x_oh[ll,jj-1]-int_x)**2)
#             ry.append((y_oh[ll,jj-1]-int_y)**2)
#             t.append(dtime[jj-1]-int_t)
#             #print('hop',ll, jj, indx_oh[ll,jj])
#             #print(int_t, int_indx, int_x, int_y)
#             #print(dtime[jj-1], indx_oh[ll,jj-1], x_oh[ll,jj-1], y_oh[ll,jj-1])     
#             #print(t[-1], rx[-1], ry[-1])   
#             #ax.append((x_oh[ll,jj]-x_oh[ll,jj-1])**2) corrected
#             #ay.append((y_oh[ll,jj]-y_oh[ll,jj-1])**2) corrected
#             #print(jj)

#             if hop>1:
#                 ax.append((x_oh[ll,jj]-x_oh[ll,jj-1])**2) #previous
#                 ay.append((y_oh[ll,jj]-y_oh[ll,jj-1])**2) #previous
#                 #print(ll,jj,(x_oh[ll,jj]-x_oh[ll,jj-1])**2 )
#                 #print(x_oh[ll,jj],x_oh[ll,jj-1])
#                 #print(dtime[jj-1],int_t)
#                 t_hop.append(dtime[jj-1]-int_t)

            
#             int_indx= indx_oh[ll,jj]
#             int_x= x_oh[ll,jj]
#             int_y= y_oh[ll,jj]
#             int_z= z_oh[ll,jj]
#             int_t=dtime[jj]
#             #print(hop,int_t)

#             #print(indx_oh[ll,jj], x_oh[ll,jj], x_oh[ll,jj-1], dtime[jj],int_t)
#     nhop= nhop+hop      
#     #print(ll,hop)
#     rx.append((x_oh[ll,-1]-int_x)**2)
#     ry.append((y_oh[ll,-1]-int_y)**2)
#     rz.append((z_oh[ll,-1]-int_z)**2)
#     t.append(dtime[-1]-int_t)
    
#     D1x[ll]=(np.mean(rx))/(2*np.mean(t))
#     D1y[ll]=(np.mean(ry))/(2*np.mean(t))
#     D1[ll]=(np.mean(rx)+np.mean(ry))/(4*np.mean(t))

# dmean= np.mean(D1)
# dstd= np.std(D1)
# std_err= dstd/np.sqrt(nrep)

# D2x=(np.mean(ax))/(2*np.mean(t_hop))
# D2y=(np.mean(ay))/(2*np.mean(t_hop))
# D2=(np.mean(ax)+np.mean(ay))/(4*np.mean(t_hop))
# D=dmean+D2


# print(nrep, D1)
# print('nohops:',dmean, std_err)
# print('hops:',D2)
# print('Total:',D)
# print(nhop)
# #print(rx)
# #print(t)


In [7]:
len(t_hop)+nrep

310

In [8]:
dmean= np.mean(D1)
dstd= np.std(D1)
std_err= dstd/np.sqrt(nrep)
dmean, dstd, std_err

(0.9627078511609803, 0.08564989947796063, 0.02211470895216539)

In [9]:
c2=0.9627078511609803
c4= 0.8756260083700356
c6=0.738363956584734

In [10]:
np.mean(ax)

2.187798605251942

In [11]:
np.mean(ay)

2.3108687306547138

In [12]:
len(ax)

310

In [13]:
len(ay)

310

In [14]:
len(t_hop)

295

In [15]:
D2x

0.11780650628446629

In [16]:
(np.mean(ax))

2.187798605251942

In [17]:
np.mean(t_hop)

9.285559322033897

In [18]:
D2y

0.12443346978416288

In [19]:
D2

0.12111998803431458